In [1]:
from flowrec.data.yambda import YambdaDataConfig, YambdaDataModule

In [2]:
cfg = YambdaDataConfig(
    use_hf=True,
    dataset_type="flat",
    dataset_size="50m",
    event_file="listens",
    min_played_ratio_pct=50,
    split_strategy="leave_one_out",
    min_user_interactions=5,
    min_item_interactions=5,
    max_history_len=128,
    batch_size=256,
    keep_raw_interactions=True,
    global_eval_target_mode="last_per_user",
    train_all_prefixes=False
)

dm = YambdaDataModule(cfg)
dm.setup()

print(dm.summary())
batch = next(iter(dm.train_dataloader()))
print(batch["history_item_ids"].shape)

/opt/anaconda3/envs/synth/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'num_users': 8804, 'num_items': 107160, 'num_train_examples': 8804, 'num_val_examples': 8528, 'num_test_examples': 8535, 'num_interactions': 7440742, 'mean_history_len': 114.1144315150578, 'mean_train_history_len': 113.92139936392549, 'mean_val_history_len': 114.16627579737336, 'mean_test_history_len': 114.26174575278266, 'post_filter_cold_items': True, 'cold_item_reference': 'observed', 'num_items_before_cold_filter': 138983, 'num_cold_items_removed': 31823, 'num_train_examples_dropped_cold': 0, 'num_val_examples_dropped_cold': 276, 'num_test_examples_dropped_cold': 269, 'num_train_history_items_removed_cold': 0, 'num_val_history_items_removed_cold': 0, 'num_test_history_items_removed_cold': 276, 'dataset': 'yambda', 'dataset_type': 'flat', 'dataset_size': '50m', 'event_file': 'listens', 'hf_start_date': None, 'hf_end_date': None, 'hf_recent_months': 2, 'split_strategy': 'leave_one_out', 'global_val_cutoff': None, 'global_test_cutoff': None, 'sid_len': None, 'item_vocab_size': 107161

In [3]:
from flowrec.item_representation.yambda import YambdaAudioEmbeddingConfig, YambdaAudioEmbeddingBuilder

store = YambdaAudioEmbeddingBuilder(
    YambdaAudioEmbeddingConfig(
        embeddings_path="../data/raw/yambda/embeddings.parquet",
        embedding_column="normalized_embed",
    )
).build()

store.save("../artifacts/item_embeddings/yambda_audio_embeddings.pt")

In [4]:
from flowrec.item_representation.content_embeddings import ItemEmbeddingStore
store = ItemEmbeddingStore.load("../artifacts/item_embeddings/yambda_audio_embeddings.pt")

In [5]:
from flowrec.item_representation.semantic_id import (
    RQVAEConfig,
    RQVAETrainConfig,
    RQVAEItemTokenizerConfig,
    RQVAEItemTokenizer,
)


# Keep only the items that remain in the final dataset after preprocessing, then
# train the RQ-VAE tokenizer on exactly those content embeddings.
store_in_data = store.subset(dm.item_ids_in_data())
print(f"{store_in_data.num_items} / {store.num_items} items remain in the final dataset")

tokenizer = RQVAEItemTokenizer(
    RQVAEItemTokenizerConfig(
        rqvae=RQVAEConfig(
            input_dim=store_in_data.embeddings.shape[1],
            latent_dim=32,
            num_codebooks=3,
            codebook_size=256,
        ),
        train=RQVAETrainConfig(epochs=50, device="cpu"),
        normalize="l2",
        collision_policy="append",
    )
)

tokenizer.fit(store_in_data.embeddings, store_in_data.item_ids)
tokenizer.save("../artifacts/tokenizers/yambda_rqvae")

semantic_id_mapping = tokenizer.semantic_id_store.to_plain_dict()

104032 / 7721749 items remain in the final dataset
Initializing RQ-VAE codebooks from data...


rqvae epoch 1/50: 100%|██████████| 97/97 [00:23<00:00,  4.09it/s, loss=0.00482, recon_loss=0.00467, vq_loss=0.00015, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0]  


{"epoch": 1, "step": 97, "train_dead_codes_l0": 23.747931, "train_dead_codes_l1": 2.331276, "train_dead_codes_l2": 3.180896, "train_loss": 0.00609, "train_recon_loss": 0.00598, "train_vq_loss": 0.000111, "val_loss": 0.00483, "val_recon_loss": 0.004677, "val_vq_loss": 0.000153}


rqvae epoch 2/50: 100%|██████████| 97/97 [00:20<00:00,  4.66it/s, loss=0.00348, recon_loss=0.00326, vq_loss=0.00022, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0] 


{"epoch": 2, "step": 194, "train_dead_codes_l0": 0.0, "train_dead_codes_l1": 0.010361, "train_dead_codes_l2": 0.186502, "train_loss": 0.004086, "train_recon_loss": 0.003902, "train_vq_loss": 0.000184, "val_loss": 0.003591, "val_recon_loss": 0.00337, "val_vq_loss": 0.000221}


rqvae epoch 3/50: 100%|██████████| 97/97 [00:20<00:00,  4.68it/s, loss=0.00306, recon_loss=0.0028, vq_loss=0.000265, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0] 


... (11 outputs left out)

rqvae epoch 9/50: 100%|██████████| 97/97 [00:22<00:00,  4.31it/s, loss=0.0021, recon_loss=0.00174, vq_loss=0.000362, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0] 


{"epoch": 9, "step": 873, "train_dead_codes_l0": 0.0, "train_dead_codes_l1": 0.0, "train_dead_codes_l2": 0.0, "train_loss": 0.002092, "train_recon_loss": 0.001735, "train_vq_loss": 0.000358, "val_loss": 0.002061, "val_recon_loss": 0.001706, "val_vq_loss": 0.000355}


rqvae epoch 10/50:  94%|█████████▍| 91/97 [00:20<00:01,  4.39it/s, loss=0.00202, recon_loss=0.00165, vq_loss=0.000371, dead_codes_l0=0, dead_codes_l1=0, dead_codes_l2=0]


KeyboardInterrupt: 

In [ ]:
# Attach the trained semantic IDs to the datamodule that was already built and
# preprocessed above, instead of constructing a second instance from a saved
# semantic_ids.json path.
dm.add_semantic_ids(semantic_ids=semantic_id_mapping)

batch = next(iter(dm.train_dataloader()))
print(batch["history_token_ids"].shape)
print(batch["target_token_ids"].shape)